<p style="text-align:center;">
<img src="https://github.com/digital-futures-academy/DataScienceMasterResources/blob/main/Resources/datascience-notebook-header.png?raw=true"
     alt="DigitalFuturesLogo"
     style="float: center; margin-right: 10px;" />
</p>

# Digital Futures Python Challenges
### Pandas – String operations

- The first part of this notebook will give a brief introduction to some of the Pandas techniques.
- The second part will contain exercises to practice the techniques.
- A separate workbook with the solutions is provided in the same directory.
- The workbook might contain materials you have not yet come across, or are not comfortable with. If so, you can consult the other workbooks and come back to this.


In [1]:
import pandas as pd

## 1. Explanatory Materials
### 1.1 Why use string methods?

In [ ]:
## While you might imagine that data science is mainly about numbers, sometimes you will need to show off some tricks
## and manipulate string type data. This can be quite time consuming and the beauty is that you never know what is to
## come, but this notebook is to prepare you for some parts of it. So, without further ado, let's look at string
## manipulation with Pandas!



In [2]:
## You, of course, can get string manipulation with methods such as comprehension:
data = ['peter', 'Paul', 'MARY', 'gUIDO']
[s.capitalize() for s in data]


['Peter', 'Paul', 'Mary', 'Guido']

In [3]:
result = []
for s in data:
    result.append(s.capitalize())

result

['Peter', 'Paul', 'Mary', 'Guido']

In [4]:
## But it has some limitations, such as handling null values:

data2 = ['peter', 'Paul', None, 'MARY', 'gUIDO']
[s.capitalize() for s in data2]

AttributeError: 'NoneType' object has no attribute 'capitalize'

In [7]:
## Luckily, Pandas has a built-in 'str' attribute that makes vectorised string operations exempt from the error above,
## and also more compact, readable and elegant. All we need to do is to call the series with the '.str' attribute
## inserted before the string operation:
names = pd.Series(data2, dtype='string')
names.str.capitalize() # series


0    Peter
1     Paul
2     <NA>
3     Mary
4    Guido
dtype: string

Besides working this time, the method also produced another side-effect. Take a look at the dtype, it is 'string' instead of 'object'! We won't delve into the differences of these two types here. An important difference to note is the enhanced quality control resulting from using 'string' instead of 'object' as 'object' is capable of mixing different datatypes while 'string' will only accept this data. You can read more about this topic in the Pandas documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/text.html

In [ ]:
## It also is important to note that the above series was created with a string method, hence its specific dtype.
## Nevertheless, you can use vectorised string manipulations even when in 'object' format:



### 1.2 Basic-Python-like string methods

In [15]:
## Many string methods are essentially the same as the ones known from base Python. So you can do hands on experiments
## to see your favourites in action!
customers = pd.Series([' MARY SMITH', 'PATRICA JOHNSON', 'LINDA williams', None, 'barbara jones'])
customers

0         MARY SMITH
1    PATRICA JOHNSON
2     LINDA williams
3                NaN
4      barbara jones
dtype: str

In [16]:
customers.str.strip().str.lower()

0         mary smith
1    patrica johnson
2     linda williams
3                NaN
4      barbara jones
dtype: str

In [17]:
## Some will return numbers:

customers.str.strip().str.len()

0    10.0
1    15.0
2    14.0
3     NaN
4    13.0
dtype: float64

In [18]:
## Some booleans
customers.str.strip().str.lower().str.contains('son')


0    False
1     True
2    False
3    False
4    False
dtype: bool

In [19]:
## Some lists:
customers.str.split()


0         [MARY, SMITH]
1    [PATRICA, JOHNSON]
2     [LINDA, williams]
3                   NaN
4      [barbara, jones]
dtype: object

In [20]:
## And more importantly, you can combine different methods:
try:
    customers.str.strip().title()
except AttributeError as error:
    print('AttributeError: ', error)


AttributeError:  'Series' object has no attribute 'title'


In [21]:
## Whoops! Not like that... you need to put the '.str' attribute before each string method, otherwise Python will
## think you want to apply the method on the entire series. And it won't like it.

clean = customers.str.strip().str.title()
clean

0         Mary Smith
1    Patrica Johnson
2     Linda Williams
3                NaN
4      Barbara Jones
dtype: str

In [25]:
## An interesting way to combine things is to use '.split()' with '.get()'. The former we have seen in action and the
## latter will return the indexed element of the created list. For example, we can return the first word of each
## string:
clean.str.split().str.get(1)


0       Smith
1     Johnson
2    Williams
3         NaN
4       Jones
dtype: object

In [26]:
## Even funkier, we can create a new dataframe by specifying the expand argument as 'True':

clean.str.split(expand = True).rename(columns = {
    0: 'first_name', 1: 'last_name'
})

,first_name,last_name
0,Mary,Smith
1,Patrica,Johnson
2,Linda,Williams
3,NaN,NaN
4,Barbara,Jones


In [29]:
## Besides the above, it is also possible to create indicator variables – which sounds odd but it's basically one-hot
## encoding (see Module 3 on linear regression) – and is called by the '.get_dummies()' method:
features = pd.Series(['Trailers,Deleted Scenes', 'Trailers', 'Commentaries,Behind the Scenes'])
features.str.get_dummies(sep = ',')


,Behind the Scenes,Commentaries,Deleted Scenes,Trailers
0,0,0,1,1
1,0,0,0,1
2,1,1,0,0


There are, of course, many more methods available. To see what's missing from here (an awful lot!), check the Pandas documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/text.html#method-summary

### 1.3 String methods utilising regex

It is possible that you will need to use more sophisticated methods to clean / organise your data. In some of these situations, you might need to resort to regular expressions. This section will discuss this topic super generally as I am nowhere near to be an expert on regex... but nobody really is. If somebody tells you that they are, you should know that this person is lying and is not to be trusted. What I would recommend is to check this cheat sheet and then try to work out problems using it: https://www.dataquest.io/wp-content/uploads/2019/03/python-regular-expressions-cheat-sheet.pdf.

You should also check the fabulous regular expressions exercises created by my colleague (who is not an expert either).

In [30]:
## We can use '.replace()' with regular expressions. Let's say we have a series of codes that start with a string of
## varying length of letters followed by a number and we only want to retain the numbers. Not sure if this is the most
## creative or lifelike example, or one that really requires regex, but it is what it is. So, let's see how you can
## solve this issue:

codes = pd.Series(['POL123', 'CLM45', 'POL9876', 'Q7'])


In [31]:
codes.str.replace('POL', 'CEL')

0     CEL123
1      CLM45
2    CEL9876
3         Q7
dtype: str

In [32]:
codes.str.replace('[A-za-z]+', '', regex=True)

0     123
1      45
2    9876
3       7
dtype: str

In [33]:
## Other methods that take regex include '.match()':
codes.str.match(r'POL\d+')

0     True
1    False
2     True
3    False
dtype: bool

In [37]:
## '.count()'
codes.str.count(r'\d')


0    3
1    2
2    4
3    1
dtype: int64

In [56]:
## '.extract()'
codes.str.extract(r'(?P<prefix>[A-Z]+)(?P<number>\d+)')


,prefix,number
0,POL,123
1,CLM,45
2,POL,9876
3,Q,7


And many more, but as for my part, this is more than I signed up for. So, please refer to the Pandas documentation or Jake VanderPlas's amazing 'Python Data Science Handbook': https://jakevdp.github.io/PythonDataScienceHandbook/.

### 1.4 Concatenation

In [39]:
## Pandas also offers concatenation, so if you would like to, you can create a giant string of all elements in a
## series / column using '.cat()':

clean.str.cat(sep = ', ')

'Mary Smith, Patrica Johnson, Linda Williams, Barbara Jones'

In [46]:
## Or, you can even concatenate with another list:
first = pd.Series(['MARY', 'PATRICIA', 'LINDA'])
first.str.cat(['SMITH', 'JOHNSON', 'WILLIAMS'], sep=' ')

0          MARY SMITH
1    PATRICIA JOHNSON
2      LINDA WILLIAMS
dtype: str

In [52]:
## And of course, with other series (in this example, it is going to be itself):
last = pd.Series(['SMITH', 'JOHNSON', 'WILLIAMS'])
first.str.cat(last, sep='_') + '@sakilacustomer.org'

0          MARY_SMITH@sakilacustomer.org
1    PATRICIA_JOHNSON@sakilacustomer.org
2      LINDA_WILLIAMS@sakilacustomer.org
dtype: str

### 1.5 Indexing and slicing

In [40]:
## Strings are iterable objects, and it only makes sense that Pandas offers a nice way to carry out element-wise
## indexing and slicing in a series or dataframe. Slicing works just the same as in base Python (just don't forget to
## add the '.str' in your code):

clean.str[0:3]

0    Mar
1    Pat
2    Lin
3    NaN
4    Bar
dtype: str

In [42]:
## You can reverse the sentences... for fun, let's say:

clean.str[::-1]

0         htimS yraM
1    nosnhoJ acirtaP
2     smailliW adniL
3                NaN
4      senoJ arabraB
dtype: str

In [45]:
## You can also combine the method with others. Let's say, in the 'codes' series, we want to know if the last three
## characters are all numbers:

codes.str[-3:].str.isdigit()

0     True
1    False
2     True
3    False
dtype: bool

### 1.6 Mapping

In [47]:
## Whilst it is not really related to string operations and is a super versatile and cool technique, we decided to
## include mapping in this workbook – like it wasn't long enough already.
## Mapping takes two arguments, the first is a function, the second one is an iterable object, and it returns a new
## iterable where the function has been applied on elements of the old iterable.
## To start illustrating the point, let's see a string method carried out with 'map()' instead of '.str'
list(map(lambda x: x.lower(), first))


['mary', 'patricia', 'linda']

In [48]:
## True, true, it is not very elegant but it works. What happens is we define an anonymous function with 'lambda' (if
## you need to refresh your memory on this, I recommend the functions exercises in this directory) that returns its
## input in lower case, then we define the input as the 'serstr' series. Python iterates through each element of the
## series and applies '.lower()' on them. Finally, we need the 'list()' function wrapped around the thing, otherwise,
## we would get an iterable object like so:
map(lambda x: x.lower(), first)


In [49]:
## Speaking of 'list()', we can also use built-in Python functions when mapping:

list(map(len, first))

[4, 8, 5]

In [50]:
## Similarly, you can use your predefined functions too with 'map()':
def abba(x):
    return f"ABBA {x}"

list(map(abba, first))


['ABBA MARY', 'ABBA PATRICIA', 'ABBA LINDA']

In [ ]:
## Yes, I only defined such a silly function to return 'ABBA'. Now go and listen to 'Super Trouper'!

In [53]:
## Finally, it is important to note that 'map()' is able to take functions with multiple variables and a corresponding
## number of iterables:
list(map(lambda a, b: a+ '_' +b, first, last))


['MARY_SMITH', 'PATRICIA_JOHNSON', 'LINDA_WILLIAMS']

In [ ]:
## The above with a predefined function:



## 2. Exercises

The following section contains exercises based on the above. You should try to do them on your own, but if you get stuck or would like to check whether what you did is correct, there is a workbook with the solutions available at the same directory (some problems might be tackled in various ways, the solutions would only include one of these).

For the exercises, the pokemon dataset is used.

In [ ]:
pokemon_dataset = "https://github.com/digital-futures-academy/DataScienceMasterResources/raw/main/Resources/pokemon.csv"
df = pd.read_csv(pokemon_dataset)
pd.set_option('display.max_columns', None)
df.head()

##### 1.) Have you ever wondered if any pokemon's name is actually a palindrome (it reads the same way backwards)? Me neither. But I am quite curious now, so let's return all pokemon (if any) whose name is a palindrome!

##### 2.) You realise how 'wannabe hacker' the current column names are. Using 'df.columms', rename them with a space between the words and each word capitalised.

##### 3.) You discover it was a mistake. Change it back (never mind the 'Unnamed: 0' column)!

##### 4.) Dr Oak wants to create a table that breaks down for each pokemon its different growth rates (true or not, we are assuming that 'Medium Slow' is not slow in a medicore way but it's either slow or medium). Unfortuantely, Dr Oak doesn't know how to use Python so he needs your help. Create a table to help him out and make sure the correct pokedex number is displayed as index!

##### 5.) The entire marketing team is on holiday and you were trusted to come up with a title for all the episodes of the next blockbuster series. Instead of relying on common sense, you decide to create an algorithm that solves all movie titles for the rest of eternity. The task is to return a string that concatenates the last word of the 'growth_rate' column, the 'status', the string: 'is the ultimate' followed by the 'name' and the first word of the 'species' column – so you would get strings such as 'Fluctuating Normal is the ultimate Bulbasaur Fire' (which is obviously not a real case but just an illustration). Make sure that the titles are in the appropriate (title) format.

##### 6.) Because the studio is short on money, they only want titles for pokemons with names longer than 20 character. Return the appropriate titles.

##### 7.) The studio realised that nobody in the office speaks Japanese. To save ink, they decided to exclude the Japanese characters from the 'japanese_name' column and only retain what's inside the brackets (without the brackets). Help them out!

##### 8.) The new pokedecks will be lit! Each card will contain an 'Overkill' stat. This is calculated by taking 'attack' to the power of 'defense'. Create a new column with the overkill stat!

##### 9.) Dr Oak wants to include poke-maths as a class in the local gym. He needs to include an extra column in the table that reads: 'pokemon.name life expectancy = total_points ** 2 / (egg_cycles * attack * 1.66) =' and then displays the value rounded to one decimal. The pokemon.name is, of course, the name of the pokemon in the actual rows. So a possible line could be
##### 'Bulbasaur life expectancy = total_points ** 2 / (egg_cycles * attack * 1.66) = 38.7'
##### And since Dr Oak still hasn't learnt Python, you have to write the script. Good luck!

##### +1.) An etymologist – don't know why, don't ask me, I'm just following orders here – visits the gym and needs some data to pump up his new study with useless info. He is curious about the length of distinct words in a pokemon's name. You should write a code that returns the desired information, e.g. 'Mega Venasaur' would return the list [4, 8] while 'Calyrex' would be [7]. Create a new column with the word-length-count!
##### Hint: you should use list comprehension!